# Reconocimiento de entidades nombradas (NER) con spaCy: fundamentos y casos de negocio

**Curso:** Procesamiento de Lenguaje Natural · Unidad 4 — Etiquetado de secuencias y NER  
**Institución:** Universidad Tecnológica de Bolívar  
**Integra:** los antiguos `ner_introduction.ipynb` y `ner_business.ipynb`  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook introduce el **reconocimiento de entidades nombradas (NER)**: encontrar en un texto los nombres de personas, organizaciones y lugares, y clasificarlos. Primero usa el modelo preentrenado en español de **spaCy** sobre una oración de ejemplo para ver qué produce un *pipeline* listo para usar: entidades, lemas, categorías gramaticales y árbol de dependencias. Después lo aplica a casos de negocio —noticias financieras, correos para un CRM y menciones de marca—, lo combina con **reglas** (`Matcher`) y con el **análisis de dependencias** para extraer relaciones, y **mide** sus errores cuando el texto se aleja del estilo de noticias con el que se entrenó.

**Objetivo:** convertir texto no estructurado en datos estructurados con un modelo preentrenado, saber qué reconoce y qué no, y decidir cuándo basta con reglas y cuándo hay que entrenar un NER propio.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Gobierno y legal — Personas y lugares en noticias

- **Problema:** una entidad quiere indexar las noticias por persona y lugar mencionados.
- **Entrada:** texto de las noticias.
- **Cómo se aplica el NER preentrenado de spaCy:**
  1. `es_core_news_sm` detecta PER, LOC y ORG en cada noticia.
  2. displaCy permite revisar rápido si las entidades son correctas.
  3. Los lemas agrupan variantes de una misma mención.
- **Salida:** índice de entidades por noticia.
- **Decisión:** búsqueda y alertas por persona o lugar.

### Finanzas y servicio al cliente — Entidades en noticias de mercado

- **Problema:** filtrar las noticias relevantes para un analista.
- **Entrada:** titulares económicos.
- **Cómo se aplica el NER preentrenado de spaCy:**
  1. NER identifica empresas (ORG) y lugares (LOC).
  2. El análisis de dependencias muestra qué hizo cada empresa (sujeto–verbo).
  3. Se descartan titulares sin empresas de interés.
- **Salida:** empresa, acción y lugar por titular.
- **Decisión:** tablero de noticias por empresa.

### Finanzas y servicio al cliente — Relaciones comerciales

- **Problema:** vigilar fusiones, adquisiciones e inversiones entre empresas.
- **Entrada:** titulares financieros.
- **Cómo se aplica NER + reglas + dependencias:**
  1. El NER de spaCy marca las organizaciones.
  2. El `Matcher` busca el patrón ORG + verbo (`adquirir`, `invertir`) + ORG.
  3. El análisis de dependencias extrae la tripleta sujeto–verbo–objeto.
- **Salida:** tripletas como (empresa A, adquiere, empresa B).
- **Decisión:** grafo de relaciones para los analistas.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install pandas spacy
python -m spacy download es_core_news_md
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `spacy` | Carga del modelo en español y procesamiento del texto |
| `spacy.displacy` | Visualización de entidades y del árbol de dependencias en Jupyter |
| `spacy.matcher.Matcher` | Reglas sobre tokens (lemas, categorías, tipos de entidad) |
| `pandas` | Tablas de entidades y de evaluación |
| `collections.defaultdict` | Agrupar menciones por tipo |

**Modelo:** `es_core_news_md` (~40 MB), entrenado con noticias en español.

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta clásica**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Business & Data Understanding | Escenario de negocio al inicio de cada caso (secciones 5–9) |
| Data | Oración de ejemplo, titulares, correos, publicaciones y frases ficticias |
| Data Preprocessing | Sección 4: tokenización, lemas y categorías gramaticales que produce el pipeline |
| Pre-trained Models | Sección 2: `es_core_news_md` de spaCy |
| Predictions | Secciones 3–9: entidades, tablas, menciones, relaciones y tripletas |
| Evaluation & QA | Secciones 8 y 10: relaciones recuperadas y precisión/recall de entidades frente a anotaciones manuales |

El modelo de spaCy es un modelo estadístico preentrenado (red neuronal pequeña, no un LLM), por eso se ubica en la ruta clásica. Aquí solo se usa para predecir; ajustarlo con datos propios está en `training_ner.ipynb`.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | displaCy, la tabla de atributos por token y las reglas del `Matcher` hacen visibles y explicables las decisiones. |
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | Usar un modelo preentrenado es adquirir IA de terceros: hay que conocer sus 4 etiquetas, su dominio (noticias) y medir su desempeño en el dominio propio (sección 10). |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | El modelo rinde peor fuera de su dominio (correos, redes, marcas nuevas); la sección 10 lo cuantifica. |
| **IEEE 7000-2021** — Proceso modelo para abordar consideraciones éticas durante el diseño de sistemas | Los casos usan empresas y personas inventadas: un sistema de extracción no debe atribuir hechos no verificados a terceros. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | spaCy y sus modelos en español se distribuyen con licencias abiertas e información de sus corpus de entrenamiento. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. **Explicar** qué es una entidad nombrada y cuáles son las etiquetas de `es_core_news_md` (PER, ORG, LOC, MISC).
2. **Describir** el análisis por token de spaCy (lema, categoría gramatical, dependencia) e **interpretar** un árbol de dependencias.
3. **Aplicar** el NER preentrenado para convertir texto no estructurado en una tabla (noticias y correos ficticios).
4. **Construir** reglas con `Matcher` y tripletas sujeto–verbo–objeto para extraer relaciones, verificando en los ejemplos cuántas de las relaciones esperadas se recuperan.
5. **Analizar** los errores típicos del modelo preentrenado (inicio de oración, entidades fuera de dominio) y **proponer** cuándo entrenar un NER propio (`training_ner.ipynb`).

> ℹ️ En las secciones 5 a 10, las empresas, personas y hechos son **inventados** para practicar; no describen hechos reales.

---

## 1. Configuración del entorno

In [1]:
# spaCy, su visualizador y reglas; pandas para tablas
import spacy
from spacy import displacy
from spacy.matcher import Matcher
from collections import defaultdict
import pandas as pd

---

## 2. Cargar el modelo de PLN en español

`es_core_news_md` es el modelo mediano de spaCy para español, entrenado con noticias. Incluye tokenizador, etiquetador morfológico, analizador de dependencias, lematizador y componente de NER.

In [2]:
try:
    nlp = spacy.load('es_core_news_md')
except OSError:
    raise SystemExit('No se encontró el modelo. Ejecuta: python -m spacy download es_core_news_md')

print(f'Modelo cargado: {nlp.meta["name"]} v{nlp.meta["version"]}')
print(f'Componentes del pipeline: {nlp.pipe_names}')
print(f'Tipos de entidad: {nlp.get_pipe("ner").labels}')

Modelo cargado: core_news_md v3.8.0
Componentes del pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Tipos de entidad: ('LOC', 'MISC', 'ORG', 'PER')


> **Interpretación de la salida:** los componentes se aplican en orden a cada texto y `ner` asigna las entidades. El modelo reconoce solo **cuatro tipos**: `LOC` (lugar), `MISC` (otras entidades: eventos, obras, nacionalidades), `ORG` (organización) y `PER` (persona). **No reconoce fechas ni montos**; para eso hacen falta reglas o un modelo entrenado con esas etiquetas.

---

## 3. Primer ejemplo: entidades de una oración

Una oración de estilo periodístico, el mismo tipo de texto con el que se entrenó el modelo.

In [3]:
texto = (
    "El presidente Pedro Sánchez visitó la sede de Naciones Unidas "
    "en Nueva York el 20 de septiembre."
)
# Un solo paso por el pipeline produce todas las anotaciones
doc = nlp(texto)

for ent in doc.ents:
    print(ent.text, ent.label_)

Pedro Sánchez PER
Naciones Unidas ORG
Nueva York LOC


> **Interpretación de la salida:** cada línea muestra una entidad y su etiqueta: *Pedro Sánchez* (`PER`), *Naciones Unidas* (`ORG`) y *Nueva York* (`LOC`). La fecha *20 de septiembre* **no** aparece, porque el modelo no tiene esa etiqueta.

In [4]:
# Resalta las entidades dentro del texto, con un color por tipo
displacy.render(doc, style="ent", jupyter=True)

> **Interpretación de la salida:** displaCy es una forma rápida de verificar visualmente las predicciones antes de integrar el modelo en un sistema. Fuera de Jupyter, `displacy.serve()` levanta un servidor local y `displacy.render(doc, style="ent", page=True)` genera un HTML independiente.

---

## 4. Análisis lingüístico por token y árbol de dependencias

spaCy anota cada token con su lema, categoría gramatical, relación de dependencia y rasgos de forma, en el mismo paso por el pipeline.

In [5]:
filas = [{"token": t.text, "lema": t.lemma_, "categoría": t.pos_, "etiqueta fina": t.tag_,
          "dependencia": t.dep_, "núcleo": t.head.text, "forma": t.shape_, "stopword": t.is_stop} for t in doc]
pd.DataFrame(filas)

,token,lema,categoría,etiqueta fina,dependencia,núcleo,forma,stopword
0,El,el,DET,DET,det,presidente,Xx,True
1,presidente,presidente,NOUN,NOUN,nsubj,visitó,xxxx,False
2,Pedro,Pedro,PROPN,PROPN,appos,presidente,Xxxxx,False
3,Sánchez,Sánchez,PROPN,PROPN,flat,Pedro,Xxxxx,False
4,visitó,visitar,VERB,VERB,ROOT,visitó,xxxx,False
5,la,el,DET,DET,det,sede,xx,True
6,sede,sede,NOUN,NOUN,obj,visitó,xxxx,False
7,de,de,ADP,ADP,case,Naciones,xx,True
8,Naciones,Naciones,PROPN,PROPN,nmod,sede,Xxxxx,False
9,Unidas,Unidas,PROPN,PROPN,flat,Naciones,Xxxxx,False


> **Interpretación de la salida:** cada fila es un token con su lema (*visitó* → *visitar*), su categoría gramatical universal (`PROPN` = nombre propio, `VERB`…), su relación de dependencia y la palabra de la que depende (`núcleo`). Estas anotaciones sirven para búsqueda por lemas, extracción de rasgos y, como se verá en las secciones 8 y 9, para escribir reglas y extraer relaciones.

In [6]:
# Árbol de dependencias sintácticas (en Jupyter se usa render; serve bloquearía la ejecución)
displacy.render(doc, style="dep", jupyter=True)

> **Interpretación de la salida:** el verbo raíz *visitó* gobierna al sujeto (*presidente*, relación `nsubj`) y al objeto (*sede*, `obj`); los nombres de varias palabras (*Pedro Sánchez*, *Naciones Unidas*, *Nueva York*) se unen con la relación `flat`.

---

## 5. Caso 1 — Entidades de noticias financieras

**Escenario:** el área de inteligencia de negocios necesita extraer empresas, personas y lugares de titulares financieros para alimentar una base de datos.

In [7]:
# Titulares ficticios
financial_news = [
    'Banco del Litoral reportó ganancias de 2.3 billones de pesos en el tercer trimestre.',
    'El presidente de Petróleos Andinos, Andrés Villamizar, anunció nuevas inversiones en el Caribe.',
    'Alimentos Sabana adquirió una empresa chilena por 450 millones de dólares.',
    'La Bolsa de Valores de Colombia cerró con una caída del 1.2% este martes.',
    'Aerolíneas del Sol reanudará vuelos directos a Madrid a partir del próximo mes.',
]

# Una fila por entidad: texto de origen, entidad, etiqueta y posiciones de caracteres
rows = []
for news in financial_news:
    for ent in nlp(news).ents:
        rows.append({'titular': news[:55] + '…', 'entidad': ent.text, 'etiqueta': ent.label_,
                     'inicio': ent.start_char, 'fin': ent.end_char})
df_entities = pd.DataFrame(rows)
print(df_entities.to_string(index=False))
print(f'\nEntidades extraídas: {len(df_entities)}')
print(df_entities['etiqueta'].value_counts().to_string())

                                                 titular                      entidad etiqueta  inicio  fin
El presidente de Petróleos Andinos, Andrés Villamizar, …            Petróleos Andinos      ORG      17   34
El presidente de Petróleos Andinos, Andrés Villamizar, …            Andrés Villamizar      PER      36   53
El presidente de Petróleos Andinos, Andrés Villamizar, …                       Caribe      LOC      88   94
Alimentos Sabana adquirió una empresa chilena por 450 m…             Alimentos Sabana      ORG       0   16
La Bolsa de Valores de Colombia cerró con una caída del… Bolsa de Valores de Colombia      ORG       3   31
Aerolíneas del Sol reanudará vuelos directos a Madrid a…                          Sol     MISC      15   18
Aerolíneas del Sol reanudará vuelos directos a Madrid a…                       Madrid      LOC      47   53

Entidades extraídas: 7
etiqueta
ORG     3
LOC     2
PER     1
MISC    1


> **Interpretación de la salida:** la tabla se puede filtrar y exportar a una base de datos; `inicio` y `fin` permiten resaltar la entidad en el texto original. Revisa los errores: en la ejecución de referencia el modelo **no detecta *Banco del Litoral*** y marca solo *Sol* (como `MISC`) en lugar de *Aerolíneas del Sol*. Tampoco extrae las cifras (*2.3 billones de pesos*), porque no tiene etiqueta para montos. La sección 10 mide estos errores.

---

## 6. Caso 2 — Correos corporativos para precargar un CRM

**Escenario:** el equipo comercial recibe cientos de correos al día y quiere extraer contacto, empresa y ciudad.

In [8]:
# Correos ficticios
corporate_emails = [
    'Buenos días, soy Carlos Martínez de Tecnología XYZ. Quiero agendar una reunión el próximo lunes 15 de enero.',
    'Hola, María López de Inversiones del Norte necesita hablar con ustedes sobre el contrato firmado en Bogotá.',
    'El señor Pedro Gómez de Constructora Andina solicita una cotización para el proyecto en Medellín.',
]

for email in corporate_emails:
    print(f'Correo: {email[:70]}…')
    print(f'  Entidades: {[(e.text, e.label_) for e in nlp(email).ents]}\n')

Correo: Buenos días, soy Carlos Martínez de Tecnología XYZ. Quiero agendar una…
  Entidades: [('Buenos días', 'LOC'), ('Carlos Martínez de Tecnología XYZ', 'PER'), ('Quiero', 'MISC')]

Correo: Hola, María López de Inversiones del Norte necesita hablar con ustedes…
  Entidades: [('María López de Inversiones del Norte', 'PER'), ('Bogotá', 'LOC')]

Correo: El señor Pedro Gómez de Constructora Andina solicita una cotización pa…
  Entidades: [('Pedro Gómez de Constructora Andina', 'PER'), ('Medellín', 'LOC')]



> **Interpretación de la salida:** idealmente `PER` → contacto, `ORG` → empresa y `LOC` → ciudad. Fuera del estilo de noticias aparecen errores típicos: *Buenos días* como lugar, *Quiero* como `MISC` (palabras en mayúscula al **inicio de oración**), y la persona unida a su empresa en una sola entidad (*Carlos Martínez de Tecnología XYZ*). Para un CRM real habría que ajustar el modelo con correos anotados y añadir reglas.

---

## 7. Caso 3 — Menciones de marcas en redes sociales

**Escenario:** mercadeo quiere saber qué organizaciones y personas se mencionan junto a la marca.

In [9]:
# Publicaciones ficticias
social_posts = [
    'El nuevo celular de TecnoNova está superando a Lumina Mobile en Colombia. #tech',
    'Petróleos Andinos y Oceánica Energy anunciaron una alianza estratégica para exploración offshore.',
    'MensajeríaYa superó a Domicilios.com en descargas durante el último trimestre.',
    'La ministra Laura Mejía firmó un acuerdo con el Banco Interamericano de Desarrollo.',
]

brand_mentions = defaultdict(list)
for post in social_posts:
    for ent in nlp(post).ents:
        brand_mentions[ent.label_].append(ent.text)
for label, mentions in brand_mentions.items():
    print(f'[{label}] {mentions}')

[LOC] ['TecnoNova', 'Lumina Mobile', 'Colombia']
[MISC] ['Petróleos Andinos']
[ORG] ['Oceánica Energy', 'Domicilios.com', 'Banco Interamericano de Desarrollo']
[PER] ['Laura Mejía']


> **Interpretación de la salida:** las marcas nuevas o inventadas (*TecnoNova*, *Lumina Mobile*) suelen salir como `LOC`: el modelo no las vio en su entrenamiento y las confunde con nombres de lugar. Agrupar por tipo solo es útil si los tipos son fiables; para monitoreo de marcas conviene una lista de marcas (`EntityRuler`) o un NER propio.

---

## 8. Relaciones comerciales con reglas (`Matcher`)

El `Matcher` define reglas sobre tokens para capturar patrones concretos, como adquisiciones o fusiones. Para cada frase se escribe a mano la relación **esperada** y se cuenta cuántas recupera cada regla.

In [10]:
# Frases ficticias y relación esperada (empresa 1, empresa 2)
business_texts = [
    'Alimentos Sabana adquirió Chocolates Andinos en una operación valorada en 300 millones.',
    'Banco del Litoral compró Financiera Rural y ampliará su red en el campo.',
    'Energías del Caribe se fusionó con Hidroeléctrica Montaña para crear un grupo energético.',
]
expected = [('Alimentos Sabana', 'Chocolates Andinos'),
            ('Banco del Litoral', 'Financiera Rural'),
            ('Energías del Caribe', 'Hidroeléctrica Montaña')]

# Regla 1: [ORG] adquirió/compró/se fusionó [con] [ORG]
matcher = Matcher(nlp.vocab)
matcher.add('REL_ORG', [[
    {'ENT_TYPE': 'ORG'},
    {'LEMMA': {'IN': ['adquirir', 'comprar', 'fusionar', 'aliarse']}},
    {'OP': '?'},
    {'ENT_TYPE': 'ORG'},
]])

print('Regla 1 (tipo de entidad ORG):')
for text in business_texts:
    doc_b = nlp(text)
    print(f'  {[doc_b[s:e].text for _, s, e in matcher(doc_b)]}  ← {text[:55]}')

Regla 1 (tipo de entidad ORG):
  []  ← Alimentos Sabana adquirió Chocolates Andinos en una ope
  []  ← Banco del Litoral compró Financiera Rural y ampliará su
  []  ← Energías del Caribe se fusionó con Hidroeléctrica Monta


> **Interpretación de la salida:** exigir `ORG` a ambos lados garantiza que los extremos sean organizaciones **solo si el NER las reconoció**. En la ejecución de referencia la regla no detecta ninguna relación: el modelo marcó *Chocolates Andinos* como `PER`, no reconoció *Banco del Litoral* y tomó las dos empresas de energía como `LOC`; además, en *“se fusionó”* el pronombre *se* se interpone entre la empresa y el verbo.

### 8.1 Una regla más robusta

Si el NER se equivoca de tipo, se puede usar la **categoría gramatical**: secuencias de nombres propios (`PROPN`) a ambos lados del verbo, admitiendo *se* y una palabra intermedia (*con*, *el*). `greedy="LONGEST"` conserva la coincidencia más larga.

In [11]:
matcher2 = Matcher(nlp.vocab)
matcher2.add('REL_PROPN', [[
    {'POS': 'PROPN', 'OP': '+'},                                   # empresa 1
    {'LOWER': 'se', 'OP': '?'},                                    # "se" opcional
    {'LEMMA': {'IN': ['adquirir', 'comprar', 'fusionar', 'aliar', 'aliarse']}},
    {'POS': {'IN': ['ADP', 'DET']}, 'OP': '?'},                    # "con", "el"… opcional
    {'POS': 'PROPN', 'OP': '+'},                                   # empresa 2
]], greedy='LONGEST')


def recuperadas(m):
    """Cuántas relaciones esperadas recupera la regla m: ambas empresas completas dentro del fragmento."""
    hits = 0
    for text, (e1, e2) in zip(business_texts, expected):
        doc_b = nlp(text)
        spans = [doc_b[s:e].text for _, s, e in m(doc_b)]
        hits += any(e1 in sp and e2 in sp for sp in spans)
        print(f'  {spans}')
    return hits

print('Regla 2 (nombres propios):')
h2 = recuperadas(matcher2)
print('\nRegla 1 (ORG):')
h1 = recuperadas(matcher)
print(f'\nRelaciones esperadas recuperadas completas → regla 1: {h1}/{len(expected)} · regla 2: {h2}/{len(expected)}')

Regla 2 (nombres propios):
  ['Alimentos Sabana adquirió Chocolates Andinos']
  ['Litoral compró Financiera Rural']
  ['Caribe se fusionó con Hidroeléctrica Montaña']

Regla 1 (ORG):
  []
  []
  []

Relaciones esperadas recuperadas completas → regla 1: 0/3 · regla 2: 1/3


> **Interpretación de la salida:** la regla por categoría gramatical encuentra las tres relaciones, aunque el NER haya fallado en el tipo. Pero en la ejecución de referencia solo una queda **completa**: en *Banco del Litoral* y *Energías del Caribe* la preposición *del* corta la secuencia de nombres propios y el fragmento empieza en *Litoral* o *Caribe*. Contar contra lo esperado, y no solo mirar si "algo coincidió", es lo que revela el problema. Una regla mejor admitiría *de/del* dentro del nombre o usaría una lista de empresas conocidas.

---

## 9. Tripletas sujeto–verbo–objeto

El análisis de dependencias permite extraer tripletas (sujeto, verbo, objeto), la base de los grafos de conocimiento.

In [12]:
def extract_svo_triplets(text: str) -> list:
    """Tripletas (sujeto, lema del verbo, objeto) con el análisis de dependencias; usa el subárbol para frases completas."""
    doc_t = nlp(text)
    triplets = []
    for token in doc_t:
        if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
            subj = next((t for t in token.lefts if t.dep_ in ('nsubj', 'nsubj:pass')), None)
            obj = next((t for t in token.rights if t.dep_ in ('obj', 'obl')), None)
            if subj is not None and obj is not None:
                frase = lambda t: ' '.join(w.text for w in t.subtree)
                triplets.append((frase(subj), token.lemma_, frase(obj)))
    return triplets

sample_sentences = [
    'Banco del Litoral anunció resultados positivos en el segundo trimestre.',
    'El gobierno firmó el contrato con Petróleos Andinos.',
    'TecnoNova lanzó un nuevo celular en Colombia.',
]
for sent in sample_sentences:
    print(f'{sent}\n  → {extract_svo_triplets(sent)}')

Banco del Litoral anunció resultados positivos en el segundo trimestre.
  → [('Banco del Litoral', 'anunciar', 'resultados positivos')]
El gobierno firmó el contrato con Petróleos Andinos.
  → [('El gobierno', 'firmar', 'el contrato con Petróleos Andinos')]
TecnoNova lanzó un nuevo celular en Colombia.
  → [('TecnoNova', 'lanzar', 'un nuevo celular')]


> **Interpretación de la salida:** `nsubj` (sujeto nominal) y `obj` (objeto directo) son relaciones universales del esquema Universal Dependencies. Usar el subárbol (`token.subtree`) devuelve la frase completa (*Banco del Litoral*, *el contrato con Petróleos Andinos*) en lugar de solo la palabra núcleo. Limitación: solo se consideran verbos raíz y un sujeto y objeto por oración.

---

## 10. Errores del modelo preentrenado y cuándo entrenar uno propio

Para pasar de "revisar a ojo" a una medida, se anotan a mano las entidades esperadas de los titulares de la sección 5 y se calculan precisión y *recall* por entidad (coincidencia exacta de texto y etiqueta).

In [13]:
# Anotación manual de los titulares ficticios (texto exacto, etiqueta)
gold = [
    {('Banco del Litoral', 'ORG')},
    {('Petróleos Andinos', 'ORG'), ('Andrés Villamizar', 'PER'), ('Caribe', 'LOC')},
    {('Alimentos Sabana', 'ORG')},
    {('Bolsa de Valores de Colombia', 'ORG')},
    {('Aerolíneas del Sol', 'ORG'), ('Madrid', 'LOC')},
]
pred = [{(e.text, e.label_) for e in nlp(t).ents} for t in financial_news]

tp = sum(len(g & p) for g, p in zip(gold, pred))
fp = sum(len(p - g) for g, p in zip(gold, pred))
fn = sum(len(g - p) for g, p in zip(gold, pred))
precision, recall = tp / (tp + fp), tp / (tp + fn)
print(f'VP={tp}  FP={fp}  FN={fn}')
print(f'Precisión={precision:.2f}  Recall={recall:.2f}  F1={2 * precision * recall / (precision + recall):.2f}')
print('\nEntidades esperadas que el modelo no recuperó:')
for g, p in zip(gold, pred):
    for falta in sorted(g - p):
        print('  ', falta)

VP=6  FP=1  FN=2
Precisión=0.86  Recall=0.75  F1=0.80

Entidades esperadas que el modelo no recuperó:
   ('Banco del Litoral', 'ORG')
   ('Aerolíneas del Sol', 'ORG')


> **Interpretación de la salida:** incluso en titulares, el registro más cercano al de entrenamiento, el modelo pierde entidades (empresas inventadas con *del* en el nombre) o recorta su texto. Con solo 5 titulares el número es orientativo; en un proyecto real se anotan cientos de textos del dominio.

**Errores típicos observados en este notebook y qué hacer:**

| Error | Ejemplo | Remedio |
|---|---|---|
| Palabras en mayúscula al inicio de oración tomadas como entidad | *Buenos días* (`LOC`), *Quiero* (`MISC`) | Filtrar entidades de una palabra al inicio de oración o ajustar con correos anotados |
| Entidades fuera del dominio de noticias | Marcas nuevas como `LOC` | `EntityRuler` con una lista de marcas, o NER propio |
| Persona y empresa unidas | *Carlos Martínez de Tecnología XYZ* | Ajuste con ejemplos anotados del dominio |
| Etiquetas que el modelo no tiene | Fechas, montos, equipos | Reglas o un NER propio con esas etiquetas |

**¿Cuándo entrenar un NER propio?** Cuando el dominio necesita **etiquetas nuevas** (equipos, productos, códigos) o cuando la medida de esta sección muestra un *recall* insuficiente en textos del dominio. `training_ner.ipynb` muestra cómo hacerlo.

---

## Resumen

| Técnica | Componente de spaCy | Uso en este notebook |
|---|---|---|
| Reconocimiento de entidades | `ner` | ORG/PER/LOC/MISC en una oración, noticias, correos y redes |
| Análisis por token | `morphologizer`, `lemmatizer`, `parser` | Lemas, categorías y dependencias |
| Reglas sobre tokens | `Matcher` | Relaciones comerciales |
| Análisis de dependencias | `parser` | Tripletas sujeto–verbo–objeto |
| Visualización | `displacy` | Revisión de entidades y del árbol |
| Evaluación | — | Precisión, *recall* y relaciones recuperadas frente a anotaciones manuales |